In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/__results__.html
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/__resultx__.html
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/__notebook__.ipynb
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/__output__.json
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/custom.css
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO2014/train_tfrecords/coco122.tfrecord
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO2014/train_tfrecords/coco13.tfrecord
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO2014/train_tfrecords/coco61.tfrecord
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO2014/train_tfrecords/coco97.tfrecord
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO2014/train_tfrecords/coco109.tfrecord
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO2014/train_tfrecords/coco84.tfrecord
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO2014/train_tfrecords/coco35.tfrecord
/kaggle/input/notebooks/fpooyan/coco2014-tfrecs/COCO201

In [1]:
!git clone https://github.com/RUCAIBox/POPE.git
!git clone https://github.com/nickjiang2378/vlm-hallucinations.git

Cloning into 'POPE'...
remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 332 (delta 12), reused 11 (delta 10), pack-reused 310 (from 1)
Receiving objects: 100% (332/332), 25.41 MiB | 16.40 MiB/s, done.
Resolving deltas: 100% (90/90), done.
Cloning into 'vlm-hallucinations'...
remote: Enumerating objects: 386, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 386 (delta 51), reused 42 (delta 42), pack-reused 285 (from 1)
Receiving objects: 100% (386/386), 24.12 MiB | 24.24 MiB/s, done.
Resolving deltas: 100% (151/151), done.


In [2]:
%cd /kaggle/working/POPE
!python main.py \
    --seg_path ./segmentation/coco_ground_truth_segmentation.json \
    --sample_num 3 --img_num 500 --dataset coco \
    --save_path /kaggle/working/pope_output/

/kaggle/working/POPE


In [3]:
!kaggle datasets init -p /kaggle/working/pope_output/
# edit the generated dataset-metadata.json title/id
!kaggle datasets create -p /kaggle/working/pope_output/

Data package template written to: /kaggle/working/pope_output/dataset-metadata.json
Default slug detected, please change values before uploading


In [4]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
import os, json
from collections import defaultdict

COCO_BASE = "/kaggle/input/datasets/nadaibrahim/coco2014"
ANN_DIR = os.path.join(COCO_BASE, "captions/annotations")

ann_path = os.path.join(ANN_DIR, "instances_val2014.json")

with open(ann_path) as f:
    coco = json.load(f)

print("images:", len(coco["images"]))            # expect 40504
print("annotations:", len(coco["annotations"]))  # expect ~291875
print("categories:", len(coco["categories"]))    # expect 80

images: 40504
annotations: 291875
categories: 80


In [14]:
import os, json
from collections import defaultdict

COCO_BASE = "/kaggle/input/datasets/nadaibrahim/coco2014"
ANN_DIR = os.path.join(COCO_BASE, "captions/annotations")

ann_path = os.path.join(ANN_DIR, "instances_val2014.json")

with open(ann_path) as f:
    coco = json.load(f)

print("images:", len(coco["images"]))            # expect 40504
print("annotations:", len(coco["annotations"]))  # expect ~291875
print("categories:", len(coco["categories"]))    # expect 80

images: 40504
annotations: 291875
categories: 80


In [15]:
import os, json

COCO_BASE = "/kaggle/input/datasets/nadaibrahim/coco2014"  # adjust to your actual mount path

# locate files once
def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

ann_path = find_file(COCO_BASE, "instances_val2014.json")
img_dir = os.path.dirname(find_file(COCO_BASE, "COCO_val2014_000000000139.jpg") or "")

with open(ann_path) as f:
    coco = json.load(f)

# build category_id -> name lookup
cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}

# build image_id -> set(object names) lookup — this is your ground truth
from collections import defaultdict
image_to_objects = defaultdict(set)
for ann in coco["annotations"]:
    image_to_objects[ann["image_id"]].add(cat_id_to_name[ann["category_id"]])

# build image_id -> filename lookup (you'll need this to open the actual image)
image_id_to_filename = {img["id"]: img["file_name"] for img in coco["images"]}

print(len(image_to_objects), "images with ground-truth objects")
print(list(image_to_objects.items())[0])

40137 images with ground-truth objects
(558840, {'bottle', 'dining table', 'cup', 'spoon', 'hot dog', 'person'})


### Pope Setup

In [16]:
!git clone https://github.com/RUCAIBox/POPE.git
%cd POPE
!python main.py \
    --seg_path ./segmentation/coco_ground_truth_segmentation.json \
    --sample_num 3 --img_num 500 --dataset coco \
    --save_path /kaggle/working/pope_output/

Cloning into 'POPE'...
remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 332 (delta 12), reused 11 (delta 10), pack-reused 310 (from 1)
Receiving objects: 100% (332/332), 25.41 MiB | 18.35 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/kaggle/working/POPE/POPE


In [17]:
import os
print(os.listdir("/kaggle/working/pope_output/coco"))

['coco_co_occur.json', 'coco_ground_truth_objects.json', 'coco_pope_adversarial.json', 'coco_pope_popular.json', 'coco_pope_random.json']


### Step 3

In [18]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [19]:
!git clone https://github.com/nickjiang2378/vlm-hallucinations.git

Cloning into 'vlm-hallucinations'...
remote: Enumerating objects: 386, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 386 (delta 51), reused 42 (delta 42), pack-reused 285 (from 1)
Receiving objects: 100% (386/386), 24.12 MiB | 24.48 MiB/s, done.
Resolving deltas: 100% (151/151), done.


In [20]:
!ls vlm-hallucinations/metric/
!cat vlm-hallucinations/metric/chair.py | head -50

chair.pkl  chair.py
'''
Copied from: https://github.com/LisaAnne/Hallucination/blob/master/utils/chair.py

Modified by: Maxlinn

1. adapt calculation of CHAIR-i and CHAIR-s for Python3, supports for both json and jsonl file input.
2. integrate synonyms.txt to make the script standalone.
3. remove machine-translation based metrics BLEU-n, CIDEr, ROGUE
4. add new metric Recall, which represents the node words(i.e. lemmas of objects) coverage overall.
5. add pickle cache mechanism to make it fast for repetitive evaluations.
'''


import os
import sys
import nltk
import json
from pattern.en import singularize
import argparse
import tqdm
import pickle
from collections import defaultdict


# copied from: https://github.com/LisaAnne/Hallucination/blob/master/data/synonyms.txt
synonyms_txt = '''
person, girl, boy, man, woman, kid, child, chef, baker, people, adult, rider, children, baby, worker, passenger, sister, biker, policeman, cop, officer, lady, cowboy, bride, groom, male, female, guy, t

In [21]:
with open("vlm-hallucinations/metric/chair.py") as f:
    content = f.read()

# extract everything between the triple quotes after "synonyms_txt = '''"
import re
match = re.search(r"synonyms_txt = '''(.*?)'''", content, re.DOTALL)
synonyms_txt = match.group(1)
print(len(synonyms_txt.splitlines()), "lines")  # should be ~80

81 lines


In [24]:
def build_synonym_dict(synonyms_txt: str) -> dict:
    synonym_dict = {}
    for line in synonyms_txt.strip().split("\n"):
        words = [w.strip() for w in line.split(",") if w.strip()]
        if not words:
            continue
        canonical = words[0]
        for w in words:
            synonym_dict[w] = canonical
    return synonym_dict

synonym_dict = build_synonym_dict(synonyms_txt)
print(len(synonym_dict), "synonym entries")
print(synonym_dict.get("labrador"), synonym_dict.get("pup"), synonym_dict.get("dog"))

403 synonym entries
dog dog dog


In [26]:
from nltk import pos_tag, word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
coco_categories = set(cat_id_to_name.values())  # from Step 1

def extract_nouns(caption: str) -> list[str]:
    tokens = word_tokenize(caption.lower())
    tagged = pos_tag(tokens)
    nouns = [lemmatizer.lemmatize(word) for word, tag in tagged if tag in ("NN", "NNS")]
    return nouns

def map_to_coco_objects(nouns: list[str], synonym_dict: dict) -> set[str]:
    """synonym_dict maps e.g. 'puppy' -> 'dog'. Falls back to direct match."""
    mapped = set()
    for noun in nouns:
        if noun in coco_categories:
            mapped.add(noun)
        elif noun in synonym_dict:
            mapped.add(synonym_dict[noun])
    return mapped

# quick test
test_caption = "A dog sits next to a wooden bench in a park. A frisbee lies nearby."
nouns = extract_nouns(test_caption)
print(nouns)

['dog', 'bench', 'park']


In [30]:
def extract_coco_objects_direct(caption: str, synonym_dict: dict, coco_categories: set) -> set:
    tokens = word_tokenize(caption.lower())
    lemmatized = [lemmatizer.lemmatize(t) for t in tokens]
    mapped = set()
    for tok in lemmatized:
        if tok in coco_categories:
            mapped.add(tok)
        elif tok in synonym_dict:
            mapped.add(synonym_dict[tok])
    return mapped

In [32]:
def map_to_coco_objects(nouns: list[str], synonym_dict: dict, coco_categories: set) -> set[str]:
    mapped = set()
    for noun in nouns:
        if noun in coco_categories:
            mapped.add(noun)
        elif noun in synonym_dict:
            mapped.add(synonym_dict[noun])
    return mapped

# test end to end
test_caption = "A dog sits next to a wooden bench in a park. A frisbee lies nearby."
nouns = extract_coco_objects_direct(test_caption, synonym_dict, coco_categories)
objects = map_to_coco_objects(nouns, synonym_dict, coco_categories)
print("nouns:", nouns)
print("mapped COCO objects:", objects)

nouns: {'dog', 'frisbee', 'bench'}
mapped COCO objects: {'dog', 'frisbee', 'bench'}


In [33]:
from nltk import pos_tag, word_tokenize

test_caption = "A dog sits next to a wooden bench in a park. A frisbee lies nearby."
tokens = word_tokenize(test_caption.lower())
tagged = pos_tag(tokens)
print(tagged)

[('a', 'DT'), ('dog', 'NN'), ('sits', 'VBZ'), ('next', 'JJ'), ('to', 'TO'), ('a', 'DT'), ('wooden', 'JJ'), ('bench', 'NN'), ('in', 'IN'), ('a', 'DT'), ('park', 'NN'), ('.', '.'), ('a', 'DT'), ('frisbee', 'JJ'), ('lies', 'VBZ'), ('nearby', 'RB'), ('.', '.')]


In [34]:
nouns = extract_nouns(test_caption)
print("extracted nouns:", nouns)
print("is 'frisbee' in nouns?", "frisbee" in nouns)
print("is 'frisbee' in coco_categories?", "frisbee" in coco_categories)

extracted nouns: ['dog', 'bench', 'park']
is 'frisbee' in nouns? False
is 'frisbee' in coco_categories? True


### To compare ground truth and model

In [ ]:
def label_caption(image_id: int, caption: str, image_to_objects: dict, synonym_dict: dict, coco_categories: set) -> list[dict]:
    """Returns a list of {object, real} rows for one captioned image."""
    mentioned = extract_coco_objects_direct(caption, synonym_dict, coco_categories)
    ground_truth = image_to_objects.get(image_id, set())
    
    rows = []
    for obj in mentioned:
        rows.append({
            "image_id": image_id,
            "object": obj,
            "real": obj in ground_truth
        })
    return rows

# test with a fake caption for now (real ones come from Track A later)
sample_id = coco["images"][0]["id"]
test_caption = "A dog sits next to a wooden bench in a park. A frisbee lies nearby."
rows = label_caption(sample_id, test_caption, image_to_objects, synonym_dict, coco_categories)
for r in rows:
    print(r)